In [ ]:
"""
If needed we match our environment installations

import sys
!{sys.executable} -m pip freeze > requirements.txt
"""

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import huggingface_hub

from huggingface_hub import hf_hub_download
from huggingface_hub import login
from transformers import pipeline
from datasets import load_dataset
from datasets import Dataset

/home/henry/miniconda3/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
login()

## Dataset

### retrieve Tweets-Dataset
Source: https://huggingface.co/datasets/zeroshot/twitter-financial-news-sentiment

Structure:
The Twitter Financial News dataset is an English-language dataset containing an annotated corpus of finance-related tweets. This dataset is used to classify finance-related tweets for their sentiment.

The dataset holds 11,932 documents annotated with 3 labels:

sentiments = {
    "LABEL_0": "Bearish", 
    "LABEL_1": "Bullish", 
    "LABEL_2": "Neutral"
}  

The data was collected using the Twitter API. The current dataset supports the multi-class classification task.

In [ ]:
# Download first:
"""
path = hf_hub_download(
    repo_id="zeroshot/twitter-financial-news-sentiment",
    filename="sent_train.csv",
    repo_type="dataset"
)
"""

In [3]:
# train-, validation- and test-set -> Use validation set for testing the model
tweets_df = pd.read_csv("../data/datasets--zeroshot--twitter-financial-news-sentiment/snapshots/ccbe24de388e287beb92dd393a335c376b350ac3/sent_train.csv")
tweets_df.head(10)

,text,label
0,$BYND - JPMorgan reels in expectations on Beyo...,0
1,$CCL $RCL - Nomura points to bookings weakness...,0
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",0
3,$ESS: BTIG Research cuts to Neutral https://t....,0
4,$FNKO - Funko slides after Piper Jaffray PT cu...,0
5,$FTI - TechnipFMC downgraded at Berenberg but ...,0
6,$GM - GM loses a bull https://t.co/tdUfG5HbXy,0
7,$GM: Deutsche Bank cuts to Hold https://t.co/7...,0
8,$GTT: Cowen cuts to Market Perform,0
9,$HNHAF $HNHPD $AAPL - Trendforce cuts iPhone e...,0


In [4]:
"""
the labels need to be correctly mapped to the model's expected labels. The model expects the following labels: 
positive, neutral, negative
and we have:
0 bearish, 1 bullish, 2 neural
"""
label_map = {
    0: "negative",
    1: "positive",
    2: "neutral"
}
tweets_df["label"] = tweets_df["label"].map(label_map)
tweets_df

,text,label
0,$BYND - JPMorgan reels in expectations on Beyo...,negative
1,$CCL $RCL - Nomura points to bookings weakness...,negative
2,"$CX - Cemex cut at Credit Suisse, J.P. Morgan ...",negative
3,$ESS: BTIG Research cuts to Neutral https://t....,negative
4,$FNKO - Funko slides after Piper Jaffray PT cu...,negative
...,...,...
9538,The Week's Gainers and Losers on the Stoxx Eur...,neutral
9539,Tupperware Brands among consumer gainers; Unil...,neutral
9540,vTv Therapeutics leads healthcare gainers; Myo...,neutral
9541,"WORK, XPO, PYX and AMKR among after hour movers",neutral


### retrieve FIQA Dataset
Citation:
@misc{xie2024finbenholisticfinancialbenchmark,
      title={FinBen: A Holistic Financial Benchmark for Large Language Models}, 
      author={Qianqian Xie and Weiguang Han and Zhengyu Chen and Ruoyu Xiang and Xiao Zhang and Yueru He and Mengxi Xiao and Dong Li and Yongfu Dai and Duanyu Feng and Yijing Xu and Haoqiang Kang and Ziyan Kuang and Chenhan Yuan and Kailai Yang and Zheheng Luo and Tianlin Zhang and Zhiwei Liu and Guojun Xiong and Zhiyang Deng and Yuechen Jiang and Zhiyuan Yao and Haohang Li and Yangyang Yu and Gang Hu and Jiajia Huang and Xiao-Yang Liu and Alejandro Lopez-Lira and Benyou Wang and Yanzhao Lai and Hao Wang and Min Peng and Sophia Ananiadou and Jimin Huang},
      year={2024},
      eprint={2402.12659},
      archivePrefix={arXiv},
      primaryClass={cs.CL},
      url={https://arxiv.org/abs/2402.12659}, 
}

Structure:
- _id: ID of the data point
- sentence: The sentence
- target: The target of the sentiment
- aspect: The aspect of the sentiment
- score: The sentiment score
- type: The type of the data point (headline or post)


In [ ]:
"""
import os
from datasets import load_dataset, load_from_disk

data_dir = "/home/henry/GitHub_Projects/AI-Visualization/data"
os.makedirs(data_dir, exist_ok=True)

# download to this folder
ds = load_dataset("TheFinAI/en-fiqa-absa", cache_dir=data_dir)

print("Dataset downloaded to:", local_dataset_path)
print(ds)
"""

Saving the dataset (1/1 shards): 100%|██████████| 117/117 [00:00<00:00, 26276.16 examples/s]

Dataset downloaded to: /home/henry/GitHub_Projects/AI-Visualization/data/en-fiqa-absa-local
DatasetDict({
    train: Dataset({
        features: ['_id', 'sentence', 'target', 'aspect', 'score', 'type'],
        num_rows: 822
    })
    test: Dataset({
        features: ['_id', 'sentence', 'target', 'aspect', 'score', 'type'],
        num_rows: 234
    })
    valid: Dataset({
        features: ['_id', 'sentence', 'target', 'aspect', 'score', 'type'],
        num_rows: 117
    })
})


In [15]:
# training-, valid- and test-set

base_dir = "../data/datasets--TheFinAI--en-fiqa-absa/snapshots/2484314781c82018dc97057c3d9165b390815d30/data"

#train_path = "../data/TheFinAI___en-fiqa-absa/default/0.0.0/2484314781c82018dc97057c3d9165b390815d30/en-fiqa-absa-train.arrow"
test_path = "../data/TheFinAI___en-fiqa-absa/default/0.0.0/2484314781c82018dc97057c3d9165b390815d30/en-fiqa-absa-test.arrow"
#valid_path = "../data/TheFinAI___en-fiqa-absa/default/0.0.0/2484314781c82018dc97057c3d9165b390815d30/en-fiqa-absa-valid.arrow"

#print("train exists?", os.path.exists(train_path))
print("test exists?", os.path.exists(test_path))
#print("valid exists?", os.path.exists(valid_path))

#train_ds = Dataset.from_file(train_path)
test_ds = Dataset.from_file(test_path)
#valid_ds = Dataset.from_file(valid_path)

#train_df = train_ds.to_pandas()
test_df = test_ds.to_pandas()
#valid_df = valid_ds.to_pandas() if valid_ds is not None else None

test exists? True


In [16]:
test_df.head(10)

,_id,sentence,target,aspect,score,type
0,645,"Britain's FTSE steadies, supported by Dixons C...",Dixons Carphone,Stock/Price Action,0.329,headline
1,651,Glaxo's ViiV Healthcare Signs China Manufactur...,Glaxo,Corporate/Sales/Partnership,0.284,headline
2,653,AstraZeneca sells Caprelsa rights to Sanofi unit,Sanofi,Corporate/Sales/Deal,0.233,headline
3,659,UPDATE 5-Barclays Chairman McFarlane axes CEO ...,Barclays,Corporate/Strategy,-0.229,headline
4,665,Miners Meltdown as BHP to Rio Tinto Sink in Co...,Rio Tinto,Stock/Price Action/Bearish/Bearish Behaviour,-0.665,headline
5,669,"FTSE led lower by M&S, GlaxoSmithKline",M&S,Market/Market/Market Trend,-0.448,headline
6,67,Morrisons faces festive sales test,Morrisons,Corporate/Sales,-0.113,headline
7,670,GKN to buy Fokker Technologies for 706 mln euros,GKN,Corporate/M&A/M&A,0.276,headline
8,671,Diageo Sells Ryder Cup Venue Gleneagles Hotel ...,Gleneagles Hotel,Corporate/Sales/Deal,0.237,headline
9,671,Diageo Sells Ryder Cup Venue Gleneagles Hotel ...,Ennismore Group,Corporate/Sales/Deal,0.198,headline


## Classifier ProsusAI/finbert

In [5]:
# get classifier
classifier = pipeline("text-classification", model="ProsusAI/finbert")

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 8095.79it/s]
BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
texts = [
    "Stocks rallied and the British pound gained.",
    "Market Uncertainty has pushed Gold prices up.",
    "The korean futures market has tripled the gains of tesla."
]

results = classifier(texts, top_k=None)
# labels as numeric array
labels = [d["label"] for d in results[0]]
softmax_array = np.array([
    [d["score"] for d in row]
    for row in results
])

print(labels)
print(softmax_array)

['positive', 'neutral', 'negative']
[[0.89836127 0.06716516 0.03447358]
 [0.89602172 0.06137564 0.04260271]
 [0.94580281 0.0288094  0.02538782]]


## Speed	
3	preds
17	seconds
5,66666666666667	seconds per prediction (1 call for every prediction)
    
12000	samples
68000	total seconds
1133,33333333333	total mins
18,8888888888889	total hours?????


In [7]:
# for a whole batch
def predict_batch(texts):
    preds = classifier(texts, batch_size=512, truncation=True, max_length=512, top_k=None)
    rows = []
    for row in preds:
        rows.append({d["label"]: float(d["score"]) for d in row})
    return rows

# example with dataframe texts
texts = tweets_df["text"].astype(str).tolist()

predictions = predict_batch(texts)

# attach to dataframe
tweets_df["predicted_scores"] = predictions
tweets_df["predicted_label"] = [
    max(row, key=row.get) for row in predictions
]

KeyboardInterrupt: 

## store predictions for visualizations and investigation

In [ ]:
tweets_df.to_csv("../data/predictions/pred_tweets.csv", index=False)